## Objective

This notebook explains predictions made by the best-performing fraud detection model (Random Forest) using:

- Built-in feature importance
- SHAP summary plots
- SHAP force plots
- Business recommendations

Best Model Selected:
- Random Forest
- AUC-PR = 0.667
- F1 Score = 0.610

In [1]:
import pandas as pd
import numpy as np

import shap
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

In [2]:
model = joblib.load(
    "../models/fraud_random_forest.pkl"
)

In [3]:
fraud_df = pd.read_csv(
    "../data/raw/processed/fraud_features.csv"
)

cols_to_drop = [
    "signup_time",
    "purchase_time",
    "device_id"
]

fraud_df = fraud_df.drop(
    columns=cols_to_drop,
    errors="ignore"
)

fraud_df = pd.get_dummies(
    fraud_df,
    columns=[
        "source",
        "browser",
        "sex"
    ],
    drop_first=True
)


X = fraud_df.drop("class", axis=1)
y = fraud_df["class"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_test.head()
print(X_test.shape)
X_test.dtypes

(30223, 16)


user_id                int64
purchase_value         int64
age                    int64
ip_address           float64
time_since_signup    float64
hour_of_day            int64
day_of_week            int64
txn_count              int64
device_txn_count       int64
source_Direct           bool
source_SEO              bool
browser_FireFox         bool
browser_IE              bool
browser_Opera           bool
browser_Safari          bool
sex_M                   bool
dtype: object

In [4]:
print(len(model.feature_importances_))
print(X_test.shape[1])

16
16


SHAP Explainability Analysis

In [ ]:
import shap

explainer = shap.Explainer(model)

Compute SHAP Values

In [ ]:
import shap

# print(shap.__version__)
# print(model.n_estimators)
# print(model.max_depth)
# X_sample = X_test.sample(
#     n=100,
#     random_state=42
# )
# shap_values = explainer.shap_values(X_sample)
X_sample = X_test.sample(
    20,
    random_state=42
)

shap_values = explainer(X_sample)

In [7]:
print(type(shap_values))

import numpy as np
print(np.array(shap_values).shape)

NameError: name 'shap_values' is not defined

Built-In Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=importance_df.head(10),
    x="Importance",
    y="Feature"
)

plt.title(
    "Top 10 Random Forest Feature Importances"
)

plt.show()

In [ ]:
plt.savefig(
    "../reports/top10_feature_importance.png",
    bbox_inches="tight"
)

SHAP Summary Plot

In [ ]:
plt.figure(figsize=(10,6))

shap.summary_plot(
    shap_values,
    X_sample,
    show=False
)

plt.tight_layout()

plt.savefig(
    "../reports/shap_summary_plot.png",
    bbox_inches="tight"
)

plt.show()

Find TP, FP, FN Cases

In [ ]:
import numpy as np

y_pred = model.predict(X_test)

tp_idx = np.where(
    (y_test == 1) &
    (y_pred == 1)
)[0][0]

fp_idx = np.where(
    (y_test == 0) &
    (y_pred == 1)
)[0][0]

fn_idx = np.where(
    (y_test == 1) &
    (y_pred == 0)
)[0][0]

print(
    f"TP={tp_idx}, FP={fp_idx}, FN={fn_idx}"
)

In [ ]:
X_force = X_test.iloc[
    [tp_idx, fp_idx, fn_idx]
]

X_force

In [ ]:
force_shap_values = explainer(
    X_force
)

True Positive Force Plot

In [ ]:
shap.plots.force(
    force_shap_values[0]
)

### True Positive Case

This transaction was correctly classified as fraud.

The features pushing the prediction toward fraud can be identified from the positive SHAP contributions shown in the force plot.

False Positive Force Plot

In [ ]:
shap.plots.force(
    force_shap_values[1]
)

### False Positive Case

This legitimate transaction was incorrectly flagged as fraud.

The force plot highlights features that contributed to the model's incorrect suspicion.

False Negative Force Plot

In [ ]:
shap.plots.force(
    force_shap_values[2]
)

### False Negative Case

This fraudulent transaction was missed by the model.

The force plot identifies factors that reduced the fraud probability enough to cause misclassification.

Extract Top SHAP Features

In [ ]:
mean_abs_shap = np.abs(
    shap_values.values
).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": X_sample.columns,
    "Mean_SHAP": mean_abs_shap
})

shap_importance = (
    shap_importance
    .sort_values(
        by="Mean_SHAP",
        ascending=False
    )
)

shap_importance.head(10)

SHAP Importance Plot

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=shap_importance.head(10),
    x="Mean_SHAP",
    y="Feature"
)

plt.title(
    "Top 10 SHAP Features"
)

plt.show()

Interpretation Section

## Interpretation of SHAP Results

### Comparison of Built-In Feature Importance and SHAP

The Random Forest built-in feature importance ranks variables according to their contribution to impurity reduction across decision trees. SHAP provides a complementary perspective by measuring the contribution of each feature to individual predictions.

The two approaches produced broadly similar rankings, indicating consistency in identifying the most influential fraud predictors. However, SHAP provides additional interpretability by showing whether each feature increases or decreases fraud probability and by quantifying its impact for individual transactions.




Business Recommendations

## Business Recommendations

### Recommendation 1: Strengthen Monitoring of Newly Created Accounts

SHAP analysis identified account age and time_since_signup as important fraud drivers. Transactions occurring shortly after account creation should be subjected to additional verification procedures such as multi-factor authentication.

### Recommendation 2: Implement Velocity-Based Fraud Detection Rules

Transaction velocity features, including transaction count and device transaction count, were among the strongest predictors of fraud. Real-time monitoring rules should trigger alerts when unusually high transaction frequencies are detected.

### Recommendation 3: Apply Risk-Based Geographic Screening

Country and location-related indicators contributed significantly to fraud predictions. Transactions originating from regions associated with elevated fraud risk should receive enhanced scrutiny and additional validation checks.
